# 02 - Baseline Training Orchestration

This notebook is the interface for baseline and light fine-tuning runs.

The implementation lives in `src/modeling/reporting.py`, so this notebook stays clean:

- no helper functions are defined here;
- training is disabled by default;
- plots are shown inline;
- final report outputs still come from `scripts/reporting/make_plots.py`.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.modeling.reporting as baseline_reporting
import src.configs as configs_module

baseline_reporting = importlib.reload(baseline_reporting)
configs_module = importlib.reload(configs_module)

PYTHON_BIN = configs_module.resolve_python_bin()
TRAIN_SCRIPT = configs_module.TRAIN_SCRIPT
MAKE_PLOTS_SCRIPT = configs_module.MAKE_PLOTS_SCRIPT
DATA_ROOT = configs_module.DEFAULT_DATA_ROOT
RESULTS_SUMMARY_PATH = configs_module.RESULTS_FINAL_TABLES_DIR / "results_summary.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

print("Project root:", PROJECT_ROOT)
print("Python:", PYTHON_BIN)
print("Training entrypoint:", TRAIN_SCRIPT)
print("Dataset root:", DATA_ROOT)


## 2.1 Execution Switches

Keep `RUN_SELECTED_EXPERIMENTS = False` unless you explicitly want to start training from this notebook.


In [ ]:
RUN_CLEAN_BASELINES = True
RUN_LIGHT_FINETUNING = False
RUN_SELECTED_EXPERIMENTS = False
SKIP_COMPLETED = True

selected_configs = baseline_reporting.get_selected_configs(
    run_clean_baselines=RUN_CLEAN_BASELINES,
    run_light_finetuning=RUN_LIGHT_FINETUNING,
)


## 2.2 Planned Runs

This table shows all baseline and light fine-tuning runs, plus whether metrics/checkpoints already exist.


In [ ]:
planned_df = baseline_reporting.build_experiment_plan()
display(planned_df)


## 2.3 Commands

Inspect the exact commands before running anything. Commands use the organized training script under `scripts/training/`.


In [ ]:
commands_df = baseline_reporting.build_command_table(
    configs=selected_configs,
    train_script=TRAIN_SCRIPT,
    python_bin=PYTHON_BIN,
    data_root=DATA_ROOT,
    skip_completed=SKIP_COMPLETED,
)

display(commands_df)


## 2.4 Optional Training Launcher

This only launches training if `RUN_SELECTED_EXPERIMENTS = True`.


In [ ]:
if not RUN_SELECTED_EXPERIMENTS:
    print("Training launcher disabled. Set RUN_SELECTED_EXPERIMENTS = True to run selected commands.")
else:
    baseline_reporting.run_selected_experiments(
        configs=selected_configs,
        train_script=TRAIN_SCRIPT,
        python_bin=PYTHON_BIN,
        data_root=DATA_ROOT,
        skip_completed=SKIP_COMPLETED,
    )


## 2.5 Completed Metrics

This reads saved metrics only. It does not retrain models.


In [ ]:
metrics_df = baseline_reporting.load_completed_metrics()

if metrics_df.empty:
    print("No completed metrics found yet.")
else:
    display(
        metrics_df[
            [
                "group",
                "model",
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "loss",
                "best_epoch",
                "batch_size",
                "num_workers",
                "run_name",
            ]
        ].style.format(
            {
                "accuracy": "{:.4f}",
                "precision_macro": "{:.4f}",
                "recall_macro": "{:.4f}",
                "f1_macro": "{:.4f}",
                "loss": "{:.4f}",
            }
        )
    )


## 2.6 Official Baseline Summary

The official table is created with `scripts/reporting/make_plots.py` and saved as `results/tables/final/results_summary.csv`.


In [ ]:
try:
    results_summary_df = baseline_reporting.load_results_summary(RESULTS_SUMMARY_PATH)
    baseline_summary_df = baseline_reporting.clean_baseline_summary(results_summary_df)

    display(
        baseline_summary_df[
            [
                "model",
                "accuracy",
                "precision",
                "recall",
                "macro_f1",
                "run_name",
                "summary_source",
            ]
        ].style.format(
            {
                "accuracy": "{:.4f}",
                "precision": "{:.4f}",
                "recall": "{:.4f}",
                "macro_f1": "{:.4f}",
            }
        )
    )
    print("Official summary table:", RESULTS_SUMMARY_PATH)
except FileNotFoundError as error:
    baseline_summary_df = pd.DataFrame()
    print(error)
    print(f"Run: {PYTHON_BIN} {MAKE_PLOTS_SCRIPT}")


## 2.7 Plots

The plotting functions are implemented in `src/modeling/reporting.py`. The notebook only displays them.


In [ ]:
if "baseline_summary_df" not in globals() or baseline_summary_df.empty:
    print("No clean baseline summary available to plot.")
else:
    fig = baseline_reporting.plot_clean_baseline_scores(baseline_summary_df)
    plt.show()


In [ ]:
if "baseline_summary_df" not in globals() or baseline_summary_df.empty:
    print("No clean baseline summary available to plot.")
else:
    fig = baseline_reporting.plot_precision_recall_tradeoff(baseline_summary_df)
    plt.show()


In [ ]:
if "metrics_df" not in globals() or metrics_df.empty:
    print("No completed metrics available to plot.")
else:
    fig = baseline_reporting.plot_baseline_vs_light_finetuning(metrics_df)
    plt.show()


## 2.8 Notes

- Use this notebook to inspect or launch baseline/fine-tuning runs.
- Use `scripts/reporting/make_plots.py` for final report tables and saved figures.
- Keep hyperparameter decisions based on validation metrics, not test metrics.
